# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets (@id and names):
print("Available record sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '(no name)')}")

# Example: Select first record set for demonstration
record_set_id = dataset.metadata.record_sets[0]['@id'] if dataset.metadata.record_sets else None
if record_set_id:
    print(f"\nFields for record set '{record_set_id}':")
    record_set_meta = next((rs for rs in dataset.metadata.record_sets if rs['@id'] == record_set_id), None)
    if record_set_meta:
        for field in record_set_meta.get('fields', []):
            name = field.get('name', '')
            print(f"  - Field @id: {field['@id']}, Name: {name}")
else:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List of record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

print(f"Extracting data from record sets: {record_sets}\n")
for rec_id in record_sets:
    try:
        recs = list(dataset.records(record_set=rec_id))
        dataframes[rec_id] = pd.DataFrame(recs)
        print(f"Loaded {len(recs)} records from record set {rec_id}")
    except Exception as e:
        print(f"Could not load records for {rec_id}: {e}")

# Display overview for the first record set with records
main_rs = None
for rec_id in record_sets:
    if rec_id in dataframes and not dataframes[rec_id].empty:
        main_rs = rec_id
        break

if main_rs:
    print(f"\nColumns in {main_rs}:\n{dataframes[main_rs].columns.tolist()}")
    display(dataframes[main_rs].head())
else:
    print("No records available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations such as removing outliers and grouping.

In [ ]:
import numpy as np

# Choose the numeric and group fields by their @id or column name in the extracted dataframe
# You might need to adjust these depending on what is present in the dataset.
if main_rs:
    df = dataframes[main_rs]
    print(f"DataFrame shape: {df.shape}")
    # Try to auto-select a likely numeric field (first float/int column)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        # If no numeric field is found, print columns and skip
        print("No numeric fields found. Columns are:", df.columns.tolist())
    else:
        threshold = df[numeric_field].mean() # set threshold as mean for demo
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std()!=0 else 1)
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a candidate categorical field (if present)
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped means of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No main record set to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_rs and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(data=dataframes[main_rs], x=numeric_field, kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.show()
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=dataframes[main_rs], x=group_field, y=numeric_field)
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using `mlcroissant`, explored its available record sets using their `@id`s, extracted records into Pandas DataFrames, and performed example EDA steps including filtering, normalization, grouping, and visualization.

You can extend the analysis by exploring additional record sets, fields, or applying domain-specific filters relevant to your molecular biology or proteomics workflows.